In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
import time
import math
from tqdm import tqdm  # Live progress bar
import matplotlib.pyplot as plt # Added for graphing

# --- CONFIGURATION ---
DATASET_ROOT = r"D:\Rail-Vision-Root\data\blurred_sharp\blurred_sharp"
BLUR_DIR = os.path.join(DATASET_ROOT, "blurred")
SHARP_DIR = os.path.join(DATASET_ROOT, "sharp")

MODEL_NAME = "kavya_deblur_real"
SAVE_DIR = "runs/restoration_logs"
IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 100 

# --- METRIC CALCULATOR (PSNR) ---
def calculate_psnr(img1, img2):
    """Calculates 'Accuracy' (PSNR) for image restoration."""
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100.0
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

# --- DATASET ---
class RealRestorationDataset(Dataset):
    def __init__(self, blur_dir, sharp_dir):
        self.blur_dir = blur_dir
        self.sharp_dir = sharp_dir
        self.filenames = sorted(os.listdir(blur_dir))
        
        # Verify valid pairs
        self.valid_files = []
        for f in self.filenames:
            if os.path.exists(os.path.join(sharp_dir, f)):
                self.valid_files.append(f)
        
        print(f"   ✅ Dataset Ready: Found {len(self.valid_files)} pairs.")

    def __len__(self):
        return len(self.valid_files)

    def __getitem__(self, idx):
        file_name = self.valid_files[idx]
        
        # Load Images
        blur_path = os.path.join(self.blur_dir, file_name)
        sharp_path = os.path.join(self.sharp_dir, file_name)
        
        blur_img = cv2.imread(blur_path)
        sharp_img = cv2.imread(sharp_path)
        
        if blur_img is None or sharp_img is None:
            return torch.zeros(3, IMG_SIZE, IMG_SIZE), torch.zeros(3, IMG_SIZE, IMG_SIZE)

        # Resize & Normalize
        blur_img = cv2.resize(blur_img, (IMG_SIZE, IMG_SIZE))
        sharp_img = cv2.resize(sharp_img, (IMG_SIZE, IMG_SIZE))
        
        input_t = torch.from_numpy(blur_img / 255.0).permute(2, 0, 1).float()
        target_t = torch.from_numpy(sharp_img / 255.0).permute(2, 0, 1).float()
        
        return input_t, target_t

# --- MODEL (UNET) ---
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1 = nn.Sequential(nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.e2 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.Conv2d(128, 128, 3, padding=1), nn.ReLU())
        self.b = nn.Sequential(nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.Conv2d(256, 128, 3, padding=1), nn.ReLU())
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.d1 = nn.Sequential(nn.Conv2d(128+128, 128, 3, padding=1), nn.ReLU())
        self.d2 = nn.Sequential(nn.Conv2d(128+64, 64, 3, padding=1), nn.ReLU())
        self.out = nn.Conv2d(64, 3, 1)

    def forward(self, x):
        x1 = self.e1(x)
        p1 = self.pool(x1)
        x2 = self.e2(p1)
        p2 = self.pool(x2)
        b = self.b(p2)
        u1 = self.up(b)
        u1 = torch.cat([u1, x2], dim=1)
        d1 = self.d1(u1)
        u2 = self.up(d1)
        u2 = torch.cat([u2, x1], dim=1)
        d2 = self.d2(u2)
        return torch.sigmoid(self.out(d2))

# --- PREVIEW GENERATOR ---
def save_preview(model, dataset, device, epoch, psnr_score):
    model.eval()
    with torch.no_grad():
        inp, targ = dataset[0] 
        inp_batch = inp.unsqueeze(0).to(device)
        pred = model(inp_batch)
        
        img_in = (inp.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        img_out = (pred.squeeze().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        img_gt = (targ.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        
        combined = np.hstack((img_in, img_out, img_gt))
        
        # Add PSNR score to the image itself so judges see it
        cv2.putText(combined, f"Epoch {epoch} | Quality: {psnr_score:.2f} dB", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        cv2.putText(combined, "Input", (10, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.putText(combined, "Restored", (270, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        
        os.makedirs(SAVE_DIR, exist_ok=True)
        cv2.imwrite(f"{SAVE_DIR}/preview_epoch_{epoch}.jpg", combined)

# --- GRAPH GENERATOR (NEW ADDITION) ---
def plot_learning_curves(history_loss, history_psnr):
    """Generates a professional dual-axis training graph."""
    epochs = range(1, len(history_loss) + 1)
    
    # Set style
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Plot Loss (Red)
    color = '#d62728' # Red
    ax1.set_xlabel('Epochs', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Loss (MSE)', color=color, fontsize=12, fontweight='bold')
    ax1.plot(epochs, history_loss, color=color, linewidth=2, linestyle='--', label='Training Loss')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, alpha=0.3)

    # Plot PSNR (Green)
    ax2 = ax1.twinx()
    color = '#2ca02c' # Green
    ax2.set_ylabel('Quality (PSNR dB)', color=color, fontsize=12, fontweight='bold')
    ax2.plot(epochs, history_psnr, color=color, linewidth=2.5, label='Restoration Quality')
    ax2.tick_params(axis='y', labelcolor=color)

    # Title & Layout
    plt.title("Kavya's Restoration Model: Training Progress", fontsize=14, fontweight='bold')
    fig.legend(loc="upper right", bbox_to_anchor=(0.85, 0.85))
    
    # Save it
    save_path = os.path.join(SAVE_DIR, "kavya_training_graph.png")
    plt.savefig(save_path, dpi=300)
    print(f"✅ Training Graph Saved: {save_path}")
    plt.close()

# --- TRAINING LOOP ---
def main():
    print(f"\n{'='*60}")
    print(f"🚀 INITIALIZING KAVYA'S RESTORATION AI (Real Data Mode)")
    print(f"{'='*60}")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✅ Hardware Accelerator: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
    
    if not os.path.exists(BLUR_DIR):
        print("❌ ERROR: Data not found.")
        return

    dataset = RealRestorationDataset(BLUR_DIR, SHARP_DIR)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    model = SimpleUNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss() 
    
    print(f"🎯 Target: Maximizing PSNR (Image Quality). >25dB is good.\n")

    # --- HISTORY TRACKERS ---
    history_loss = []
    history_psnr = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0
        total_psnr = 0
        
        # 🟢 LIVE PROGRESS BAR
        loop = tqdm(loader, leave=True)
        loop.set_description(f"Epoch [{epoch}/{EPOCHS}]")
        
        for batch_in, batch_target in loop:
            batch_in, batch_target = batch_in.to(device), batch_target.to(device)
            
            optimizer.zero_grad()
            output = model(batch_in)
            loss = criterion(output, batch_target)
            loss.backward()
            optimizer.step()
            
            # Compute Stats
            batch_psnr = calculate_psnr(output, batch_target)
            total_loss += loss.item()
            total_psnr += batch_psnr.item()
            
            # UPDATE BAR: Show Loss AND Accuracy (PSNR)
            loop.set_postfix(loss=loss.item(), quality_dB=batch_psnr.item())
        
        # Epoch Stats
        avg_loss = total_loss / len(loader)
        avg_psnr = total_psnr / len(loader)
        
        # STORE HISTORY
        history_loss.append(avg_loss)
        history_psnr.append(avg_psnr)
        
        # Save preview with Score
        if epoch % 5 == 0 or epoch == 1:
            save_preview(model, dataset, device, epoch, avg_psnr)
            
    # --- SAVE MODEL & GRAPH ---
    os.makedirs("models", exist_ok=True)
    torch.save(model.state_dict(), f"models/{MODEL_NAME}.pth")
    
    # GENERATE GRAPH
    plot_learning_curves(history_loss, history_psnr)
    
    print(f"\n✅ Training Complete. Final Quality: {avg_psnr:.2f} dB")

if __name__ == "__main__":
    main()


🚀 INITIALIZING KAVYA'S RESTORATION AI (Real Data Mode)
✅ Hardware Accelerator: NVIDIA GeForce RTX 4060 Laptop GPU
   ✅ Dataset Ready: Found 1151 pairs.
🎯 Target: Maximizing PSNR (Image Quality). >25dB is good.



Epoch [100/100]: 100%|██████████| 288/288 [00:39<00:00,  7.34it/s, loss=0.000768, quality_dB=31.1]


✅ Training Graph Saved: runs/restoration_logs\kavya_training_graph.png

✅ Training Complete. Final Quality: 28.62 dB


In [4]:
from ultralytics import RTDETR
import pandas as pd
import matplotlib.pyplot as plt
import os

# --- CONFIGURATION ---
DATA_YAML = r"D:\Rail-Vision-Root\data\wagon_dataset\data.yaml"
MODEL_NAME = "rail_vision_transformer_large"
PROJECT_DIR = "runs/detect"  # Default Ultralytics save dir

# --- GRAPH GENERATOR FUNCTION ---
def plot_rtdetr_metrics(save_dir):
    """
    Reads the 'results.csv' generated by RT-DETR and creates a professional graph.
    """
    csv_path = os.path.join(save_dir, 'results.csv')
    
    if not os.path.exists(csv_path):
        print(f"⚠️ Could not find log file at {csv_path}. Graph skipped.")
        return

    # Load Data (Ultralytics CSVs sometimes have extra spaces in headers)
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    epochs = df['epoch']
    
    # Extract Metrics (Handling slight name variations in YOLO versions)
    map50 = df['metrics/mAP50(B)']
    
    # Loss: Prefer Box Loss, fallback to others if needed
    if 'train/box_loss' in df.columns:
        loss = df['train/box_loss']
        loss_label = 'Box Loss'
    else:
        loss = df['train/giou_loss'] # Older versions
        loss_label = 'GIoU Loss'

    # --- PLOT SETUP ---
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # 1. Plot Accuracy (Blue)
    color = '#1e3a8a' # Dark Blue
    ax1.set_xlabel('Epochs', fontsize=12, fontweight='bold')
    ax1.set_ylabel('mAP@50 (Accuracy)', color=color, fontsize=12, fontweight='bold')
    ax1.plot(epochs, map50, color=color, linewidth=2.5, label='Validation mAP@50')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.set_ylim(0, 1.0)
    ax1.grid(True, alpha=0.3)

    # 2. Plot Loss (Red) - On Right Axis
    ax2 = ax1.twinx()
    color = '#dc2626' # Red
    ax2.set_ylabel(f'Training {loss_label}', color=color, fontsize=12, fontweight='bold')
    ax2.plot(epochs, loss, color=color, linewidth=2, linestyle='--', label='Train Loss')
    ax2.tick_params(axis='y', labelcolor=color)

    # Title & Save
    plt.title("RT-DETR Training Dynamics (Transformer Backbone)", fontsize=14, fontweight='bold')
    fig.legend(loc="center right", bbox_to_anchor=(0.85, 0.5))
    
    graph_path = os.path.join(save_dir, 'masum_training_graph.png')
    plt.savefig(graph_path, dpi=300)
    print(f"✅ Training Graph Saved: {graph_path}")
    plt.close()

# --- MAIN TRAINING FUNCTION ---
def train_transformer():
    print("🚀 TRAINING RT-DETR (Transformer Based Detection)...")
    print("   ► Backbone: Vision Transformer (ViT)")
    print("   ► Advantage: Highest Accuracy for Small Defects")

    # Load Model
    model = RTDETR("rtdetr-l.pt") 
    
    # Start Training
    # We capture the 'results' object, but the CSV is saved to disk automatically
    results = model.train(
        data=DATA_YAML,
        epochs=75,          # Transformers need time to converge
        imgsz=640,          # Standard size
        device=0,           # GPU
        batch=4,            # Keep small to save VRAM
        project=PROJECT_DIR,
        name=MODEL_NAME,
        
        # --- TRANSFORMER SETTINGS ---
        optimizer='AdamW',
        lr0=0.0001,
        warmup_epochs=5,
        
        # --- AUGMENTATION ---
        mosaic=1.0,
        hsv_h=0.015,
        hsv_s=0.7, 
        hsv_v=0.4,
    )
    
    # --- AUTO-GENERATE GRAPH ---
    # Ultralytics saves to runs/detect/{name}, or increments {name}2, {name}3...
    # The 'results.save_dir' property gives us the exact folder used for THIS run.
    print(f"\n📊 Generating Training Graph in: {model.trainer.save_dir}")
    plot_rtdetr_metrics(model.trainer.save_dir)
    
    print(f"🎉 Training & Graph Generation Complete!")

if __name__ == "__main__":
    train_transformer()

🚀 TRAINING RT-DETR (Transformer Based Detection)...
   ► Backbone: Vision Transformer (ViT)
   ► Advantage: Highest Accuracy for Small Defects
New https://pypi.org/project/ultralytics/8.3.250 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.241  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Rail-Vision-Root\data\wagon_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=75, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=N

c:\Users\Kavya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\autograd\graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/75      3.52G      1.318     0.9518      1.176         20        640: 100% ━━━━━━━━━━━━ 133/133 3.1it/s 42.5s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 8.0it/s 0.9s0.1s
                   all         51        136    0.00529      0.596     0.0149    0.00345

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/75      3.52G      1.089     0.7209     0.5974         15        640: 0% ──────────── 0/133  0.3s

c:\Users\Kavya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\autograd\graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/75      3.52G      1.026     0.8089     0.7317         13        640: 100% ━━━━━━━━━━━━ 133/133 3.3it/s 40.1s0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 8.7it/s 0.8s0.1s
                   all         51        136    0.00627      0.706     0.0736      0.018

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/75      3.52G     0.5993      1.364     0.5126          8        640: 0% ──────────── 0/133  0.4s

...

      73/75      3.52G     0.2931     0.4769     0.2325         12        640: 100% ━━━━━━━━━━━━ 133/133 3.5it/s 37.8s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 9.5it/s 0.7s0.1s
                   all         51        136      0.754      0.653      0.688      0.469

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      74/75      3.52G     0.5747     0.6708     0.7122          8        640: 0% ──────────── 0/133  0.3s

c:\Users\Kavya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\autograd\graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      74/75      3.52G     0.2745     0.4992     0.2225         11        640: 100% ━━━━━━━━━━━━ 133/133 3.6it/s 37.4s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 9.3it/s 0.7s0.1s
                   all         51        136       0.76      0.629       0.69      0.472

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      75/75      3.52G     0.2545     0.5721     0.2448         13        640: 0% ──────────── 0/133  0.3s

c:\Users\Kavya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\autograd\graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      75/75      3.52G     0.2795     0.4752     0.2296          6        640: 100% ━━━━━━━━━━━━ 133/133 3.6it/s 36.7s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 9.5it/s 0.7s0.1s
                   all         51        136      0.757      0.642      0.709      0.488

75 epochs completed in 0.825 hours.
Optimizer stripped from D:\Rail-Vision-Root\src\ai_core\runs\detect\rail_vision_transformer_large\weights\last.pt, 66.2MB
Optimizer stripped from D:\Rail-Vision-Root\src\ai_core\runs\detect\rail_vision_transformer_large\weights\best.pt, 66.2MB

Validating D:\Rail-Vision-Root\src\ai_core\runs\detect\rail_vision_transformer_large\weights\best.pt...
Ultralytics 8.3.241  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95

In [3]:
pip install ultralytics pandas matplotlib

  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.3 MB 7.1 MB/s eta 0:00:02
   --------- ------------------------------ 2.6/11.3 MB 7.9 MB/s eta 0:00:02
   ----------------- ---------------------- 5.0/11.3 MB 9.1 MB/s eta 0:00:01
   -------------------------------- ------- 9.2/11.3 MB 12.1 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 13.1 MB/s  0:00:00
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)

   ---------------------------------------- 0/3 [pytz]
   ------------- -------------------------- 1/3 [tzdata]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   ------------------------